In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [6]:
weather_train = pd.read_csv("../data/raw/weather/DailyDelhiClimateTrain.csv")
weather_test = pd.read_csv("../data/raw/weather/DailyDelhiClimateTest.csv")

weather_df = pd.concat([weather_train, weather_test], ignore_index=True)

pollution_df = pd.read_csv("../data/raw/pollution/city_day.csv")

accident_df = pd.read_csv(
    "../data/raw/accidents/US_Accidents_March23.csv",
    nrows=100000,
    low_memory=False
)

In [7]:
accident_df = accident_df[
[
'Severity',
'Start_Time',
'Start_Lat',
'Start_Lng',
'Temperature(F)',
'Humidity(%)',
'Pressure(in)',
'Visibility(mi)',
'Wind_Speed(mph)',
'Weather_Condition'
]
]

In [8]:
accident_df['Start_Time'] = pd.to_datetime(accident_df['Start_Time'])

In [9]:
accident_df['year'] = accident_df['Start_Time'].dt.year

In [10]:
accident_df['month'] = accident_df['Start_Time'].dt.month

In [11]:
accident_df['day'] = accident_df['Start_Time'].dt.day

In [12]:
accident_df['hour'] = accident_df['Start_Time'].dt.hour

In [13]:
accident_df['day_of_week'] = accident_df['Start_Time'].dt.dayofweek

In [14]:
accident_df['is_weekend'] = (
    accident_df['day_of_week'] >= 5
).astype(int)

In [15]:
accident_df.head()

,Severity,Start_Time,Start_Lat,Start_Lng,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Weather_Condition,year,month,day,hour,day_of_week,is_weekend
0,3,2016-02-08 05:46:00,39.865147,-84.058723,36.9,91.0,29.68,10.0,NaN,Light Rain,2016,2,8,5,0,0
1,2,2016-02-08 06:07:59,39.928059,-82.831184,37.9,100.0,29.65,10.0,NaN,Light Rain,2016,2,8,6,0,0
2,2,2016-02-08 06:49:27,39.063148,-84.032608,36.0,100.0,29.67,10.0,3.5,Overcast,2016,2,8,6,0,0
3,3,2016-02-08 07:23:34,39.747753,-84.205582,35.1,96.0,29.64,9.0,4.6,Mostly Cloudy,2016,2,8,7,0,0
4,2,2016-02-08 07:39:07,39.627781,-84.188354,36.0,89.0,29.65,6.0,3.5,Mostly Cloudy,2016,2,8,7,0,0


In [16]:
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

In [17]:
accident_df["time_of_day"] = accident_df["hour"].apply(get_time_of_day)

In [18]:
accident_df[["hour", "time_of_day"]].head(15)

,hour,time_of_day
0,5,Morning
1,6,Morning
2,6,Morning
3,7,Morning
4,7,Morning
5,7,Morning
6,7,Morning
7,7,Morning
8,8,Morning
9,8,Morning


In [19]:
def classify_weather(condition):

    condition = str(condition)

    if "Rain" in condition:
        return "Rain"

    elif "Snow" in condition:
        return "Snow"

    elif "Fog" in condition:
        return "Fog"

    elif "Cloud" in condition:
        return "Cloudy"

    else:
        return "Clear"

In [20]:
accident_df["Weather_Category"] = accident_df["Weather_Condition"].apply(classify_weather)

In [21]:
accident_df[["Weather_Condition", "Weather_Category"]].head(15)

,Weather_Condition,Weather_Category
0,Light Rain,Rain
1,Light Rain,Rain
2,Overcast,Clear
3,Mostly Cloudy,Cloudy
4,Mostly Cloudy,Cloudy
5,Light Rain,Rain
6,Overcast,Clear
7,Overcast,Clear
8,Mostly Cloudy,Cloudy
9,Light Rain,Rain


In [22]:
from sklearn.preprocessing import LabelEncoder

weather_encoder = LabelEncoder()

accident_df["Weather_Category"] = weather_encoder.fit_transform(
    accident_df["Weather_Category"]
)

In [23]:
time_encoder = LabelEncoder()

accident_df["time_of_day"] = time_encoder.fit_transform(
    accident_df["time_of_day"]
)

In [24]:
accident_df.head()

,Severity,Start_Time,Start_Lat,Start_Lng,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Weather_Condition,year,month,day,hour,day_of_week,is_weekend,time_of_day,Weather_Category
0,3,2016-02-08 05:46:00,39.865147,-84.058723,36.9,91.0,29.68,10.0,NaN,Light Rain,2016,2,8,5,0,0,2,3
1,2,2016-02-08 06:07:59,39.928059,-82.831184,37.9,100.0,29.65,10.0,NaN,Light Rain,2016,2,8,6,0,0,2,3
2,2,2016-02-08 06:49:27,39.063148,-84.032608,36.0,100.0,29.67,10.0,3.5,Overcast,2016,2,8,6,0,0,2,0
3,3,2016-02-08 07:23:34,39.747753,-84.205582,35.1,96.0,29.64,9.0,4.6,Mostly Cloudy,2016,2,8,7,0,0,2,1
4,2,2016-02-08 07:39:07,39.627781,-84.188354,36.0,89.0,29.65,6.0,3.5,Mostly Cloudy,2016,2,8,7,0,0,2,1


In [25]:
features = [
    "Start_Lat",
    "Start_Lng",
    "Temperature(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
    "is_weekend",
    "time_of_day",
    "Weather_Category"
]

In [26]:
target = "Severity"

In [27]:
X = accident_df[features]

y = accident_df[target]

In [28]:
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

X.head()

Features Shape: (100000, 15)
Target Shape: (100000,)


,Start_Lat,Start_Lng,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),year,month,day,hour,day_of_week,is_weekend,time_of_day,Weather_Category
0,39.865147,-84.058723,36.9,91.0,29.68,10.0,NaN,2016,2,8,5,0,0,2,3
1,39.928059,-82.831184,37.9,100.0,29.65,10.0,NaN,2016,2,8,6,0,0,2,3
2,39.063148,-84.032608,36.0,100.0,29.67,10.0,3.5,2016,2,8,6,0,0,2,0
3,39.747753,-84.205582,35.1,96.0,29.64,9.0,4.6,2016,2,8,7,0,0,2,1
4,39.627781,-84.188354,36.0,89.0,29.65,6.0,3.5,2016,2,8,7,0,0,2,1
